# The no-X-ray run against the fiducial one: the nox phenotype, its analogues in the AGN-on runs, and their AGN history

`paper_ism_prediction_boxes.ipynb` found that the quenched galaxies of **s50nox** (the m50n512 run without X-ray feedback) sit on the observed dust fractions the fiducial run misses by ~1.6 dex. This notebook asks three questions of the same catalogue products (pure reads — nothing here opens a catalogue or a snapshot):

1. **Part 1 — what is a nox quenched galaxy?** The s50nox quenched population against the fiducial s50 one, quantity by quantity (age, size at fixed $M_\star$ and $z$, $\kappa_{\rm rot}$ of stars and gas, $\Sigma_{\rm e}$, dust / gas / H$_2$ fractions, sSFR, $M_{\rm BH}$), plus a mass- and anchor-matched pairing so the contrasts are not a mass-function artefact. Is the nox QG an old, extended, rotating disc?
2. **Part 2 — do the AGN-on runs contain such sources?** A Mahalanobis match in a configurable feature space (`FEATURES`): every quenched galaxy of s50 / m100 whose distance to the nox cloud is within the cloud's own `ANALOG_Q` quantile is a **nox analogue**. How many are there, and do they reproduce the full phenotype (not just the dust)?
3. **Part 3 — what AGN history do the analogues have?** The pre-quenching coupling ($w_{\rm pre}$, weak / strong), the first X-ray episode's timing (`agn_onset`), and the post-SFT exposure ($E_x$, $f_x$, $T_{\rm gas\text{-}poor}$, the gate's timing) of the analogues against the rest of the same box — is the nox-like phenotype in the AGN-on runs a *weak-coupling* population, or an *under-exposed* one (undergrown BH, late gate, $E_x \approx 0$)?

**Inputs** (all written by `paper_ism_prediction_boxes.ipynb` into `output/box_resolution/ism_prediction/`): `ism_prediction_catalogue.fits` (Part 1 cache: every galaxy with $\log M_\star > 9.5$ per box and anchor), `ism_prediction_agn_classes.csv` (Part 6a: SFT / QT, $w_{\rm pre}$, the class, the onset), `ism_prediction_xray_exposure.csv` (Part 9: the exposure integrals). **Outputs** to `output/box_resolution/noxray_analogues/` (`noxray_analogues_*`).

In [ ]:
# ── Part 0 — configuration: pure reads of the paper_ism_prediction_boxes products; the samples, the derived axes, the helpers ──
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.table import Table
from scipy.stats import mannwhitneyu, fisher_exact, wilcoxon

IN  = os.path.join(os.getcwd(), "output", "box_resolution", "ism_prediction")      # the boxes notebook's products (this notebook only reads)
OUT = os.path.join(os.getcwd(), "output", "box_resolution", "noxray_analogues")
CACHE_FITS = os.path.join(IN, "ism_prediction_catalogue.fits")                     # its Part 1 cache: every galaxy with log M* > 9.5 per (box, anchor)
CLASS_CSV  = os.path.join(IN, "ism_prediction_agn_classes.csv")                    # its Part 6a: SFT / QT, w_pre, the class, the onset
EXP_CSV    = os.path.join(IN, "ism_prediction_xray_exposure.csv")                  # its Part 9: the exposure integrals since SFT
os.makedirs(OUT, exist_ok=True)

NOX_BOX, FID_BOX = "cis50nox", "cis50"     # the comparison: the no-X-ray run vs the fiducial one (same box, resolution and selection)
REF_BOX          = "cis100"                # the large fiducial box carried alongside (same gas particle mass as m50)
ANALOG_BOXES     = ["cis50", "cis100", "cis25"]   # the AGN-on runs searched for nox analogues in Part 2 (cis25: mind its 8x lower gas-mass floor)
BOX_STYLE = {"cis100":   dict(color="#D55E00", label="m100n1024 (fiducial)"),
             "cis50":    dict(color="#009E73", label="m50n512 s50 (fiducial)"),
             "cis25":    dict(color="#0072B2", label="m25n512 (fiducial)"),
             "cis50nox": dict(color="#CC79A7", label="m50n512 s50nox (no X-ray)")}

# the m25 selection rule and the shared constants, verbatim from paper_ism_prediction_boxes Part 0
MASS_FLOOR, PASSIVE_FACTOR, NGAS_MIN, NSTAR_MIN, DUST_TO_H2_MIN = 10.0, 0.2, 21, 20, 1e-4
MASS_MIN, POOL_LOGM, HE, R_PROJ_OVER_3D, SSFR_FLOOR = 10.25, 9.5, 1.36, 0.75, 1e-13
JET_LOGMBH, JET_FEDD = 7.5, 0.2                                   # SIMBA's jet criterion at the anchor
PRE_THR_WEAK, PRE_THR_STRONG, TWO_CLASS_THR = 0.10, 0.50, 0.30    # the class cuts (the boxes notebook's agn_class is its CLASS_SCHEME "two")
CLASSES  = ["weak", "strong"]
P9_FLOOR = -6.5                                                   # zero dust sits at this log (the end point of the removal, never dropped)
XRAY_FGAS_MAX = 0.2
KEY = ["box", "snap", "gal_id"]
MSTAR_COL, SFR_COL = "mstar", "sfr"

PAPER_DPI = 300
plt.rcParams.update({"font.size": 11, "axes.titlesize": 11, "axes.labelsize": 11, "legend.fontsize": 9, "xtick.labelsize": 9.5,
                     "ytick.labelsize": 9.5, "axes.titleweight": "bold", "pdf.fonttype": 42})


def need(path, made_by):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} is missing: {made_by}")
    return path


def paper_save(fig, stem):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(OUT, f"{stem}.{ext}"), dpi=PAPER_DPI, bbox_inches="tight")
    print("figure ->", os.path.join(OUT, f"{stem}.{{png,pdf}}"))


# ── the catalogue cache -> the quenched tracks' sample (the boxes notebook's Part 1 rule, verbatim) ──
CAT = Table.read(need(CACHE_FITS, "paper_ism_prediction_boxes Part 1")).to_pandas()
CAT["box"] = [v.decode() if isinstance(v, bytes) else str(v) for v in CAT["box"]]
CAT["box"] = CAT["box"].str.strip()
_ms, _sfr = CAT["mstar"].to_numpy(float), CAT["sfr"].to_numpy(float)
with np.errstate(divide="ignore", invalid="ignore"):
    _ssfr = np.where(_ms > 0, _sfr / _ms, np.nan)
    _lm = np.log10(np.where(_ms > 0, _ms, np.nan))
_all = ((_lm > MASS_FLOOR) & (_ssfr < PASSIVE_FACTOR / (CAT["t_H_gyr"].to_numpy(float) * 1e9))
        & (CAT["ngas"].to_numpy(float) >= NGAS_MIN) & (CAT["nstar"].to_numpy(float) >= NSTAR_MIN)
        & (CAT["mdust"].to_numpy(float) >= DUST_TO_H2_MIN * CAT["mh2"].to_numpy(float)))
QM = CAT[_all].copy()
with np.errstate(divide="ignore", invalid="ignore"):
    ms = QM[MSTAR_COL].to_numpy(float); ms = np.where(ms > 0, ms, np.nan)
    QM["log_mstar"] = np.log10(QM["mstar"].to_numpy(float))
    QM["fdust"] = QM["mdust"].to_numpy(float) / ms
    QM["fh2"]   = HE * QM["mh2"].to_numpy(float) / ms
    QM["fgas"]  = QM["mgas"].to_numpy(float) / ms
    QM["ssfr"]  = np.maximum(QM[SFR_COL].to_numpy(float), 0.0) / ms
    QM["re"]    = R_PROJ_OVER_3D * QM["r_half_star"].to_numpy(float)
    QM["sig_e"] = np.where(QM["re"] > 0, 0.5 * ms / (np.pi * QM["re"] ** 2), np.nan)
    mbh = QM["mbh"].to_numpy(float)
    QM["jet"] = (mbh > 0) & (np.log10(np.where(mbh > 0, mbh, 1.0)) > JET_LOGMBH) & (QM["fedd"].to_numpy(float) < JET_FEDD) & np.isfinite(QM["fedd"].to_numpy(float))
    QM["log_mbh"] = np.where(mbh > 0, np.log10(np.where(mbh > 0, mbh, 1.0)), np.nan)
    fd, mg, md = QM["fdust"].to_numpy(float), QM["mgas"].to_numpy(float), QM["mdust"].to_numpy(float)
    QM["lfd"]   = np.where(fd > 0, np.log10(np.where(fd > 0, fd, 1.0)), P9_FLOOR)                  # zero dust at the floor
    QM["ldg"]   = np.where((md > 0) & (mg > 0), np.log10(np.where((md > 0) & (mg > 0), md / np.where(mg > 0, mg, 1.0), 1.0)), np.where(mg > 0, P9_FLOOR, np.nan))
    QM["lfgas"] = np.log10(np.where(QM["fgas"] > 0, QM["fgas"], np.nan))
    QM["lfh2"]  = np.log10(np.where(QM["fh2"] > 0, QM["fh2"], np.nan))
    QM["lsig"]  = np.log10(QM["sig_e"].to_numpy(float))
    QM["lre"]   = np.log10(np.where(QM["re"] > 0, QM["re"], np.nan))
    QM["lssfr"] = np.log10(QM["ssfr"].clip(lower=SSFR_FLOOR))
QM = QM[QM["log_mstar"] > MASS_MIN].reset_index(drop=True)
QM["gkey"] = [f"{b}_{int(s)}_{int(g)}" for b, s, g in zip(QM["box"], QM["snap"], QM["gal_id"])]

# ── the classes and the exposures merged in (left joins: a box / anchor without histories keeps NaN there) ──
CLS = pd.read_csv(need(CLASS_CSV, "paper_ism_prediction_boxes Part 6a"))
EXP = pd.read_csv(need(EXP_CSV, "paper_ism_prediction_boxes Part 9"))
QM = QM.merge(CLS[KEY + ["t_sft", "t_qt", "tau_q", "w_pre", "t_onset", "agn_onset", "agn_class3", "agn_class"]], on=KEY, how="left")
QM = QM.merge(EXP[KEY + ["dt_sft", "e_x", "e_jet", "t_gaspoor", "f_x", "r_x", "lag_gate", "gate_at_sft", "fgas_anchor"]], on=KEY, how="left")

# ── the size at fixed M* and z: log R_e fit on the FIDUCIAL quenched sample, the residual (dlre) for everyone ──
_f = QM[(QM["box"] == FID_BOX) & np.isfinite(QM["lre"]) & np.isfinite(QM["log_mstar"])]
_A = np.c_[np.ones(len(_f)), _f["log_mstar"].to_numpy(float) - 10.5, np.log10(1 + _f["z"].to_numpy(float))]
_c = np.linalg.lstsq(_A, _f["lre"].to_numpy(float), rcond=None)[0]
QM["dlre"] = QM["lre"] - (_c[0] + _c[1] * (QM["log_mstar"] - 10.5) + _c[2] * np.log10(1 + QM["z"]))
print(f"size relation (fit on the {len(_f)} {FID_BOX} quenched galaxies): log R_e = {_c[0]:.2f} {_c[1]:+.2f} (log M* - 10.5) {_c[2]:+.2f} log(1+z); dlre = the residual, for every box")


# ── helpers ──
def mw_auc(x, m1):
    """x over two populations (m1 = population 1): AUC = P(x_1 > x_0), the two-sided Mann-Whitney p, the two finite counts."""
    x = np.asarray(x, float); m1 = np.asarray(m1, bool)
    a, b = x[m1 & np.isfinite(x)], x[~m1 & np.isfinite(x)]
    if len(a) < 3 or len(b) < 3 or np.nanstd(np.concatenate([a, b])) == 0:
        return np.nan, np.nan, len(a), len(b)
    u = mannwhitneyu(a, b, alternative="two-sided")
    return float(u.statistic) / (len(a) * len(b)), float(u.pvalue), len(a), len(b)


def med3(v):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    return (np.nan, np.nan, np.nan) if not len(v) else tuple(np.percentile(v, [50, 16, 84]))


def ecdf(ax, v, **kw):
    v = np.sort(np.asarray(v, float)); v = v[np.isfinite(v)]
    if len(v):
        ax.step(np.r_[v[0], v], np.r_[0.0, np.arange(1, len(v) + 1) / len(v)], where="post", **kw)


# ── the samples ──
print(f"\nthe quenched samples (m25 rule on the catalogue columns, log M* > {MASS_MIN:g} — the boxes notebook's tracks' sample):")
print(f"  {'box':9s} {'N':>5s} {'anch':>4s} {'sat%':>5s} {'jet%':>5s} {'log M*':>7s} {'age':>5s} {'R_e':>5s} {'kap_s':>6s} {'kap_g':>6s} {'log f_d':>8s} {'log f_gas':>9s} {'classified':>10s} {'exposure':>8s}")
for b in dict.fromkeys([NOX_BOX, FID_BOX, REF_BOX] + ANALOG_BOXES):
    g = QM[QM["box"] == b]
    if not len(g):
        print(f"  {b:9s} MISSING from the cache"); continue
    print(f"  {b:9s} {len(g):5d} {g['snap'].nunique():4d} {100 * (1 - g['central'].mean()):4.0f}% {100 * g['jet'].mean():4.0f}% {g['log_mstar'].median():7.2f} {g['age'].median():5.2f} "
          f"{g['re'].median():5.2f} {g['kappa_star'].median():6.2f} {g['kappa_gas'].median():6.2f} {g['lfd'].median():8.2f} {g['lfgas'].median():9.2f} "
          f"{int(g['agn_class'].isin(CLASSES).sum()):10d} {int(np.isfinite(g['e_x']).sum()):8d}")
print(f"  (classified = weak / strong; exposure = finite E_x; {NOX_BOX} has neither — no history ladder on the share, its AGN state is the anchor jet criterion above)")

## Part 1 — the nox quenched population against the fiducial one: old, extended, rotating discs?

The two populations are the tracks' samples (m25 selection rule, $\log M_\star >$ `MASS_MIN`) of **s50nox** and **s50** — same box, same resolution, same selection, one physics switch. Three views:

* **(a) distributions** — per quantity the medians (16–84 %) of the two populations, the Mann–Whitney $p$ and the AUC (the probability a random nox galaxy exceeds a random fiducial one) $\to$ `noxray_analogues_populations.csv`; m100 is carried alongside as the large-volume fiducial reference (same gas particle mass as m50).
* **(b) the disc verdict** — the fraction of each population that is a *rotating disc* ($\kappa_{\rm rot} \geq$ `DISC_KAPPA`, stars and gas separately), *old* (age $\geq$ the fiducial median) and *extended* ($\Delta \log R_{\rm e} > 0$ at fixed $M_\star$ and $z$; the size relation is fit on the fiducial quenched sample and applied to everyone), with Fisher's $p$ per fraction.
* **(c) the mass-matched pairing** — every nox QG paired greedily with the nearest-in-$\log M_\star$ unused s50 QG **at the same anchor** (within `MATCH_DM` dex): the median paired difference and the Wilcoxon $p$ per quantity — the same contrasts with the mass function and the anchor mix held fixed.

Figure `noxray_analogues_populations`: the CDFs of the nine axes of the story (age, $\kappa_\star$, $\kappa_{\rm gas}$, $\Delta \log R_{\rm e}$, $\log \Sigma_{\rm e}$, dust and gas fractions, sSFR, $M_\star$), nox against s50 with m100 as the thin reference.

In [ ]:
# ── Part 1 — s50nox vs s50: the distributions, the disc verdict, the mass- and anchor-matched pairing ──
DISC_KAPPA = 0.5     # a "rotating disc": kappa_rot at or above this
MATCH_DM   = 0.20    # (c) a pair must agree in log M* within this [dex], at the same anchor
QTY1 = [("log_mstar", "log M*"), ("z", "redshift"), ("age", "age [Gyr]"), ("re", "R_e [kpc]"), ("dlre", "dlog R_e | M*,z"),
        ("lsig", "log Sigma_e"), ("kappa_star", "kappa_rot stars"), ("kappa_gas", "kappa_rot gas"), ("lfd", "log M_dust/M*"),
        ("ldg", "log M_dust/M_gas"), ("lfgas", "log M_gas/M*"), ("lfh2", "log 1.36 M_H2/M*"), ("lssfr", "log sSFR"), ("log_mbh", "log M_BH")]

GN, GF, G100 = (QM[QM["box"] == b] for b in (NOX_BOX, FID_BOX, REF_BOX))
rows = []

# (a) the distributions
print(f"(a) {NOX_BOX} (N = {len(GN)}) vs {FID_BOX} (N = {len(GF)}): medians (16-84 %), Mann-Whitney p, AUC = P(nox > fiducial); the {REF_BOX} median alongside")
print(f"  {'quantity':18s} {'nox: med (16-84)':>26s} {'s50: med (16-84)':>26s} {'m100':>7s} {'delta':>7s} {'p':>9s} {'AUC':>5s}")
_both = pd.concat([GN, GF]); _isnox = np.r_[np.ones(len(GN), bool), np.zeros(len(GF), bool)]
for col, lab in QTY1:
    mn, ln_, hn = med3(GN[col]); mf, lf_, hf = med3(GF[col]); m1 = med3(G100[col])[0]
    auc, p, na, nb = mw_auc(_both[col], _isnox)
    rows.append(dict(kind="distribution", quantity=col, label=lab, n_nox=na, n_fid=nb, med_nox=mn, lo_nox=ln_, hi_nox=hn,
                     med_fid=mf, lo_fid=lf_, hi_fid=hf, med_m100=m1, delta=mn - mf, p=p, auc=auc))
    print(f"  {lab:18s} {mn:9.2f} ({ln_:6.2f} {hn:6.2f}) {mf:9.2f} ({lf_:6.2f} {hf:6.2f}) {m1:7.2f} {mn - mf:+7.2f} {p:9.2g} {auc:5.2f}")

# (b) the disc verdict: the fractions and Fisher's p
FID_AGE_MED = float(np.nanmedian(GF["age"]))
FRACS = [("kappa_star", DISC_KAPPA, f"rotating disc, stars (kappa >= {DISC_KAPPA:g})"),
         ("kappa_gas",  DISC_KAPPA, f"rotating disc, gas (kappa >= {DISC_KAPPA:g})"),
         ("age",        FID_AGE_MED, f"old (age >= the fiducial median {FID_AGE_MED:.2f} Gyr)"),
         ("dlre",       0.0, "extended (above the fiducial size relation)")]
print(f"\n(b) the disc verdict — the fraction of each population that is:")
print(f"  {'criterion':46s} {'nox':>12s} {'s50':>12s} {'m100':>12s} {'Fisher p (nox vs s50)':>21s}")
for col, thr, lab in FRACS + [("central", 0.5, "a central"), ("jet", 0.5, "in jet mode at the anchor")]:
    ks, Ns = [], []
    for g in (GN, GF, G100):
        v = g[col].to_numpy(float); fin = np.isfinite(v)
        ks.append(int((v[fin] >= thr).sum())); Ns.append(int(fin.sum()))
    p = fisher_exact([[ks[0], Ns[0] - ks[0]], [ks[1], Ns[1] - ks[1]]])[1] if min(Ns[:2]) else np.nan
    rows.append(dict(kind="fraction", quantity=col, label=lab, n_nox=Ns[0], n_fid=Ns[1], med_nox=ks[0] / max(Ns[0], 1), med_fid=ks[1] / max(Ns[1], 1),
                     med_m100=ks[2] / max(Ns[2], 1), delta=ks[0] / max(Ns[0], 1) - ks[1] / max(Ns[1], 1), p=p, auc=np.nan))
    print(f"  {lab:46s} {100 * ks[0] / max(Ns[0], 1):4.0f}% ({ks[0]:4d}) {100 * ks[1] / max(Ns[1], 1):4.0f}% ({ks[1]:4d}) {100 * ks[2] / max(Ns[2], 1):4.0f}% ({ks[2]:4d}) {p:21.2g}")

# (c) the mass- and anchor-matched pairing (greedy nearest in log M*, without replacement)
_pn, _pf = [], []
for s, gs in GN.groupby("snap"):
    avail = GF[GF["snap"] == s].copy()
    for _, r in gs.sort_values("log_mstar", ascending=False).iterrows():
        if not len(avail):
            break
        d = (avail["log_mstar"] - r["log_mstar"]).abs()
        j = d.idxmin()
        if d.loc[j] <= MATCH_DM:
            _pn.append(r); _pf.append(avail.loc[j]); avail = avail.drop(index=j)
PN, PF = pd.DataFrame(_pn), pd.DataFrame(_pf)
print(f"\n(c) matched pairs: {len(PN)} of {len(GN)} nox galaxies paired to a distinct {FID_BOX} galaxy (same anchor, |dlog M*| <= {MATCH_DM:g})"
      + (f"; |dlog M*| median {np.median(np.abs(PN['log_mstar'].to_numpy() - PF['log_mstar'].to_numpy())):.3f}" if len(PN) else ""))
if len(PN) >= 8:
    print(f"  {'quantity':18s} {'median nox - fid diff':>21s} {'Wilcoxon p':>11s}")
    for col, lab in QTY1:
        d = PN[col].to_numpy(float) - PF[col].to_numpy(float); d = d[np.isfinite(d)]
        try:
            p = float(wilcoxon(d).pvalue) if len(d) >= 8 and np.any(d != 0) else np.nan
        except ValueError:
            p = np.nan
        rows.append(dict(kind="matched", quantity=col, label=lab, n_nox=len(d), n_fid=len(d), med_nox=np.nan, med_fid=np.nan,
                         med_m100=np.nan, delta=float(np.median(d)) if len(d) else np.nan, p=p, auc=np.nan))
        print(f"  {lab:18s} {np.median(d) if len(d) else np.nan:+21.2f} {p:11.2g}")

POP_CSV = os.path.join(OUT, "noxray_analogues_populations.csv")
pd.DataFrame(rows).to_csv(POP_CSV, index=False)
print(f"contrasts -> {POP_CSV}")

# ── the figure: the CDFs of the nine axes of the story ──
PANELS = [("age", "age [Gyr]"), ("kappa_star", r"$\kappa_{\rm rot}$ stars"), ("kappa_gas", r"$\kappa_{\rm rot}$ gas"),
          ("dlre", r"$\Delta \log R_{\rm e}$ at fixed $M_\star$, $z$"), ("lsig", r"$\log \Sigma_{\rm e}$ [$M_\odot$ kpc$^{-2}$]"),
          ("lfd", r"$\log M_{\rm dust}/M_\star$"), ("lfgas", r"$\log M_{\rm gas}/M_\star$"), ("lssfr", r"$\log$ sSFR [yr$^{-1}$]"), ("log_mstar", r"$\log M_\star$")]
fig, axs = plt.subplots(3, 3, figsize=(11.5, 8.6))
for ax, (col, lab) in zip(axs.ravel(), PANELS):
    ecdf(ax, G100[col], color=BOX_STYLE[REF_BOX]["color"], lw=1.2, ls=(0, (4, 2)), alpha=0.85)
    ecdf(ax, GF[col],   color=BOX_STYLE[FID_BOX]["color"], lw=2.4)
    ecdf(ax, GN[col],   color=BOX_STYLE[NOX_BOX]["color"], lw=2.8)
    r = next(rr for rr in rows if rr["kind"] == "distribution" and rr["quantity"] == col)
    ax.text(0.03, 0.96, f"p = {r['p']:.1g}\nAUC = {r['auc']:.2f}", transform=ax.transAxes, va="top", fontsize=8.5,
            bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=1.5))
    ax.set_xlabel(lab); ax.set_ylim(0, 1); ax.grid(alpha=0.25, lw=0.5)
for i in range(3):
    axs[i, 0].set_ylabel("CDF")
_h = [plt.Line2D([], [], color=BOX_STYLE[b]["color"], lw=w, ls=s, label=BOX_STYLE[b]["label"])
      for b, w, s in [(NOX_BOX, 2.8, "-"), (FID_BOX, 2.4, "-"), (REF_BOX, 1.2, (0, (4, 2)))]]
fig.legend(handles=_h, loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.015))
fig.suptitle("the quenched population without X-ray feedback vs the fiducial one (whole-galaxy catalogue quantities)", y=1.005)
fig.tight_layout(rect=(0, 0.025, 1, 1))
paper_save(fig, "noxray_analogues_populations")
plt.show()

## Part 2 — the nox analogues inside the AGN-on runs

The nox phenotype is summarised by `FEATURES` — **default: age and $\log M_{\rm dust}/M_\star$ only** (zero dust at the `P9_FLOOR`), the two cheapest axes of the phenotype: the match deliberately does *not* see rotation, size or gas content, so Part 2b can test whether they come along for free. The fuller structural set of the first version (age, $\kappa_{\rm gas}$, $\kappa_\star$, $\Delta \log R_{\rm e}$, dust, gas) is listed in the cell; a feature with poor coverage in any population is dropped and listed. Every quenched galaxy of the AGN-on boxes (`ANALOG_BOXES`) gets the Mahalanobis distance $D^2$ to the nox cloud (mean and covariance of the nox feature vectors); an **analogue** is a galaxy within the nox cloud's own `ANALOG_Q` quantile of $D^2$ — i.e. it would not stand out among the nox QGs. Reported per box:

* the analogue count and fraction (per anchor too), the feature medians of analogues / nox / the rest (does the match reproduce the whole phenotype?) $\to$ `noxray_analogues_match_medians.csv`, and the per-galaxy table (distance, flag, features, class, exposure) $\to$ `noxray_analogues_match.csv`;
* the **reverse containment** — the fraction of nox QGs inside the *fiducial* cloud's own `ANALOG_Q` region: the asymmetry says whether nox is a subset of the fiducial diversity or a genuinely displaced population.

Figure `noxray_analogues_match`: per box the (age, $\log M_{\rm dust}/M_\star$) and ($\kappa_\star$, $\Delta \log R_{\rm e}$) planes (grey = the box's quenched rest, colour = the analogues, pink = the nox cloud) and the $\sqrt{D^2}$ distributions with the threshold.

In [ ]:
# ── Part 2 — the nox analogues inside the AGN-on runs: Mahalanobis distance to the nox cloud in the feature space ──
FEATURES     = ["age", "lfd"]   # the selection: age + log M_dust/M* ONLY (the observable pair the search is built on; the rest of the
                                # phenotype is then a PREDICTION tested in Part 2b). The joint structural set of the first version:
                                # ["age", "kappa_gas", "kappa_star", "dlre", "lfd", "lfgas"]
ANALOG_Q     = 0.90      # an analogue sits within the nox cloud's own this-quantile of D^2 (it would not stand out among the nox QGs)
FEAT_MIN_COV = 0.7       # a feature finite for less than this fraction of the nox sample or of a searched box is dropped (listed)

GN = QM[QM["box"] == NOX_BOX]
FEATS = []
for f in FEATURES:
    cov = [float(np.isfinite(QM.loc[QM["box"] == b, f].to_numpy(float)).mean()) for b in [NOX_BOX] + ANALOG_BOXES]
    if min(cov) >= FEAT_MIN_COV:
        FEATS.append(f)
    else:
        print(f"  feature {f} DROPPED (coverage: " + ", ".join(f"{b} {100 * c:.0f} %" for b, c in zip([NOX_BOX] + ANALOG_BOXES, cov)) + ")")
K = len(FEATS)


def cloud(g):
    """Complete feature rows of one population -> (X, mean, inverse covariance with a small ridge)."""
    X = g[FEATS].to_numpy(float); X = X[np.isfinite(X).all(axis=1)]
    C = np.cov(X.T); C += np.eye(K) * 1e-6 * np.trace(C) / K
    return X, X.mean(axis=0), np.linalg.inv(C)


def d2_to(X, mu, Ci):
    d = np.asarray(X, float) - mu
    return np.einsum("ij,jk,ik->i", d, Ci, d)


XN, MU_N, CI_N = cloud(GN)
D2_THR = float(np.quantile(d2_to(XN, MU_N, CI_N), ANALOG_Q))
print(f"the nox cloud: {len(XN)} of {len(GN)} galaxies with all {K} features ({', '.join(FEATS)}); analogue threshold D^2 <= {D2_THR:.2f} "
      f"(the cloud's own {100 * ANALOG_Q:.0f} % quantile; a chi2_k({K}) 90 % point would be ~{K + 1.8 * np.sqrt(2 * K):.1f})")

QM["d2_nox"], QM["nox_analog"] = np.nan, False
med_rows = []
for b in ANALOG_BOXES:
    m = (QM["box"] == b).to_numpy()
    X = QM.loc[m, FEATS].to_numpy(float); ok = np.isfinite(X).all(axis=1)
    idx = QM.index[m][ok]
    QM.loc[idx, "d2_nox"] = d2_to(X[ok], MU_N, CI_N)
    QM.loc[idx, "nox_analog"] = QM.loc[idx, "d2_nox"] <= D2_THR
    g = QM[m]; an = g["nox_analog"].to_numpy(bool)
    print(f"\n{b}: {int(an.sum())} analogues of {int(ok.sum())} quenched galaxies with all features ({len(g)} quenched) = {100 * an.sum() / max(ok.sum(), 1):.1f} %")
    per = g[an].groupby("z_target").size()
    tot = g.groupby("z_target").size()
    print("  per anchor: " + (", ".join(f"z={z:g} {per.get(z, 0)}/{n}" for z, n in tot.items()) if len(tot) else "none"))
    print(f"  {'quantity':12s} {'analogues':>9s} {'nox':>8s} {'rest':>8s}    (medians: does the match reproduce the whole phenotype?)")
    for col in FEATS + ["log_mstar", "lssfr", "lsig", "z", "central", "re"]:
        ma = med3(g.loc[an, col])[0]; mr = med3(g.loc[~an, col])[0]; mn = med3(GN[col])[0]
        med_rows.append(dict(box=b, quantity=col, n_analog=int(an.sum()), med_analog=ma, med_nox=mn, med_rest=mr))
        print(f"  {col:12s} {ma:9.2f} {mn:8.2f} {mr:8.2f}")

# the reverse containment: is nox a subset of the fiducial diversity, or a displaced population?
GFC = QM[QM["box"] == FID_BOX]
XF, MU_F, CI_F = cloud(GFC)
_thr_f = float(np.quantile(d2_to(XF, MU_F, CI_F), ANALOG_Q))
_in_f = float((d2_to(XN, MU_F, CI_F) <= _thr_f).mean())
_in_n = float(QM.loc[QM["box"] == FID_BOX, "nox_analog"].mean())
print(f"\nreverse containment: {100 * _in_f:.0f} % of the nox QGs sit inside the {FID_BOX} cloud's {100 * ANALOG_Q:.0f} % region, "
      f"while {100 * _in_n:.0f} % of {FID_BOX} sits inside the nox cloud's — the asymmetry says whether nox is a displaced population or a subset of the fiducial diversity")

MED_CSV = os.path.join(OUT, "noxray_analogues_match_medians.csv")
pd.DataFrame(med_rows).to_csv(MED_CSV, index=False)
_outcols = KEY + ["z", "z_target", "log_mstar", "d2_nox", "nox_analog"] + FEATS + ["lssfr", "lsig", "central", "jet", "agn_class", "w_pre", "e_x", "f_x", "agn_onset"]
MATCH_CSV = os.path.join(OUT, "noxray_analogues_match.csv")
QM.loc[QM["box"].isin(ANALOG_BOXES), _outcols].to_csv(MATCH_CSV, index=False)
print(f"medians -> {MED_CSV}; per-galaxy distances / flags -> {MATCH_CSV}")

# ── the figure: two phenotype planes and the distance distribution per searched box ──
fig, axs = plt.subplots(len(ANALOG_BOXES), 3, figsize=(12.8, 4.0 * len(ANALOG_BOXES)), squeeze=False)
for i, b in enumerate(ANALOG_BOXES):
    g = QM[QM["box"] == b]; an = g["nox_analog"].to_numpy(bool); c = BOX_STYLE[b]["color"]
    for j, (xc, yc, xl, yl) in enumerate([("age", "lfd", "age [Gyr]", r"$\log M_{\rm dust}/M_\star$"),
                                          ("kappa_star", "dlre", r"$\kappa_{\rm rot}$ stars", r"$\Delta \log R_{\rm e}$")]):
        ax = axs[i, j]
        ax.scatter(g.loc[~an, xc], g.loc[~an, yc], s=6, color="0.75", lw=0, alpha=0.55, label="quenched, the rest", rasterized=True)
        ax.scatter(GN[xc], GN[yc], s=26, facecolor="none", edgecolor=BOX_STYLE[NOX_BOX]["color"], lw=0.9, label=BOX_STYLE[NOX_BOX]["label"])
        ax.scatter(g.loc[an, xc], g.loc[an, yc], s=17, color=c, edgecolor="0.15", lw=0.4, label="nox analogues")
        ax.set_xlabel(xl); ax.set_ylabel(yl); ax.grid(alpha=0.25, lw=0.5)
    ax = axs[i, 2]
    dn = np.sqrt(d2_to(XN, MU_N, CI_N)); db = np.sqrt(g["d2_nox"].to_numpy(float)); db = db[np.isfinite(db)]
    hi = max(float(np.percentile(db, 99)) if len(db) else 6.0, float(dn.max()) * 1.05)
    bins = np.linspace(0, hi, 36)
    ax.hist(dn, bins=bins, density=True, histtype="stepfilled", color=BOX_STYLE[NOX_BOX]["color"], alpha=0.35, label="s50nox (self-distance)")
    ax.hist(db, bins=bins, density=True, histtype="step", color=c, lw=2.0, label=BOX_STYLE[b]["label"])
    ax.axvline(np.sqrt(D2_THR), color="0.2", ls=":", lw=1.4)
    ax.text(np.sqrt(D2_THR), ax.get_ylim()[1] * 0.97, " analogue threshold", va="top", fontsize=8.2, color="0.25")
    ax.set_xlabel(r"$\sqrt{D^2}$ to the nox cloud"); ax.set_ylabel("density"); ax.grid(alpha=0.25, lw=0.5)
    axs[i, 0].set_title(f"{BOX_STYLE[b]['label']}: {int(an.sum())} nox analogues", loc="left", fontsize=10.5)
    if i == 0:
        axs[0, 0].legend(loc="lower left", fontsize=8)
        axs[0, 2].legend(loc="center right", fontsize=8)
fig.tight_layout()
paper_save(fig, "noxray_analogues_match")
plt.show()

## Part 2b — Part 1 on the analogues: what the (age, $f_{\rm dust}$) selection does and does not reproduce

The analogues are now selected on **age and $\log M_{\rm dust}/M_\star$ alone**, so every other quantity is a genuine prediction: if the (age, $f_{\rm dust}$)-selected fiducial galaxies also carry the nox rotation, sizes and gas content, the nox phenotype is one package an observer can reach through the two cheapest of its axes. Per analogue box:

* the Part-1 contrast of the **analogues against the nox population itself** — medians, Mann–Whitney $p$ and the AUC $= P(\rm analogue > nox)$ per quantity (a $p \sim 1$ / AUC $\sim 0.5$ row = a nox property the selection reproduced without asking for it; a significant row = a real difference between the fiducial look-alikes and the true no-X-ray population) $\to$ `noxray_analogues_analogue_populations.csv`, CDF grid figure `noxray_analogues_analogue_populations` (nox thick, one line per box's analogues);
* the **(age, $\log M_{\rm dust}/M_\star$) plane coloured by the dust-to-gas ratio** $\log M_{\rm dust}/M_{\rm gas}$ (`noxray_analogues_dgr_plane`): the two match axes with the physics that is *not* matched on — if the analogues sit at the nox D/G, they hold their dust the nox way (never scoured); if they are visibly bluer at the same (age, $f_{\rm dust}$), they are gas-rich systems passing through, not survivors.

In [ ]:
# ── Part 2b — Part 1 on the analogues: contrasts against nox itself, the CDF grid, the (age, f_dust) plane coloured by D/G ──
MIN_2B = 5
GNX = QM[QM["box"] == NOX_BOX]
rows2b = []
print(f"the analogues against the nox population itself (the match saw only {', '.join(FEATS)}; everything else is a prediction). AUC = P(analogue > nox)")
for b in ANALOG_BOXES:
    ga = QM[(QM["box"] == b) & QM["nox_analog"]]
    if len(ga) < MIN_2B:
        print(f"\n  {b}: {len(ga)} analogues — too few for the contrast"); continue
    print(f"\n  {b} ({len(ga)} analogues): satellites {100 * (1 - ga['central'].mean()):.0f} % (nox {100 * (1 - GNX['central'].mean()):.0f} %), "
          f"jet at the anchor {100 * ga['jet'].mean():.0f} % (nox {100 * GNX['jet'].mean():.0f} %)")
    print(f"    {'quantity':18s} {'analog med':>10s} {'nox med':>8s} {'delta':>7s} {'p':>9s} {'AUC':>5s}")
    for col, lab in QTY1:
        pooled = pd.concat([ga, GNX]); m1 = np.r_[np.ones(len(ga), bool), np.zeros(len(GNX), bool)]
        auc, p, n1, n0 = mw_auc(pooled[col], m1)
        ma, mn = med3(ga[col])[0], med3(GNX[col])[0]
        rows2b.append(dict(box=b, quantity=col, label=lab, n_analog=n1, n_nox=n0, med_analog=ma, med_nox=mn, delta=ma - mn, p=p, auc=auc))
        print(f"    {lab:18s} {ma:10.2f} {mn:8.2f} {ma - mn:+7.2f} {p:9.2g} {auc:5.2f}")
POP2B_CSV = os.path.join(OUT, "noxray_analogues_analogue_populations.csv")
pd.DataFrame(rows2b).to_csv(POP2B_CSV, index=False)
print(f"\ncontrasts -> {POP2B_CSV}")

# ── the CDF grid: nox (thick) and each box's analogues on the Part 1 panels ──
fig, axs = plt.subplots(3, 3, figsize=(11.5, 8.6))
for ax, (col, lab) in zip(axs.ravel(), PANELS):
    ecdf(ax, GNX[col], color=BOX_STYLE[NOX_BOX]["color"], lw=3.2)
    txt = []
    for b in ANALOG_BOXES:
        ga = QM[(QM["box"] == b) & QM["nox_analog"]]
        if len(ga) < MIN_2B:
            continue
        ecdf(ax, ga[col], color=BOX_STYLE[b]["color"], lw=1.8)
        r = next((rr for rr in rows2b if rr["box"] == b and rr["quantity"] == col), None)
        if r is not None:
            txt.append(f"{b.replace('cis', 'm')}: {r['auc']:.2f}")
    ax.text(0.03, 0.96, "AUC vs nox\n" + "\n".join(txt), transform=ax.transAxes, va="top", fontsize=7.8,
            bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=1.5))
    ax.set_xlabel(lab); ax.set_ylim(0, 1); ax.grid(alpha=0.25, lw=0.5)
for i in range(3):
    axs[i, 0].set_ylabel("CDF")
_h = [plt.Line2D([], [], color=BOX_STYLE[NOX_BOX]["color"], lw=3.2, label=BOX_STYLE[NOX_BOX]["label"])] + \
     [plt.Line2D([], [], color=BOX_STYLE[b]["color"], lw=1.8, label=f"{BOX_STYLE[b]['label']} analogues") for b in ANALOG_BOXES
      if (QM.loc[QM["box"] == b, "nox_analog"]).sum() >= MIN_2B]
fig.legend(handles=_h, loc="lower center", ncol=len(_h), frameon=False, bbox_to_anchor=(0.5, -0.015))
fig.suptitle(f"the analogues (selected on {', '.join(FEATS)} only) against the no-X-ray population", y=1.005)
fig.tight_layout(rect=(0, 0.025, 1, 1))
paper_save(fig, "noxray_analogues_analogue_populations")
plt.show()

# ── the (age, log f_dust) plane coloured by the dust-to-gas ratio: nox, then each box's analogues over its quenched rest ──
_pn = [(NOX_BOX, None)] + [(b, "analog") for b in ANALOG_BOXES]
_lv = pd.concat([GNX["ldg"]] + [QM.loc[(QM["box"] == b) & QM["nox_analog"], "ldg"] for b in ANALOG_BOXES])
VMIN, VMAX = np.nanpercentile(_lv, [5, 95])
fig, axs = plt.subplots(1, len(_pn), figsize=(3.9 * len(_pn), 3.9), sharex=True, sharey=True)
for ax, (b, mode) in zip(np.atleast_1d(axs), _pn):
    g = QM[QM["box"] == b]
    if mode is None:
        sc = ax.scatter(g["age"], g["lfd"], c=g["ldg"], s=13, cmap="viridis", vmin=VMIN, vmax=VMAX, lw=0)
        ax.set_title(f"{BOX_STYLE[b]['label']} (all quenched)", fontsize=9.5)
    else:
        an = g["nox_analog"].to_numpy(bool)
        ax.scatter(g.loc[~an, "age"], g.loc[~an, "lfd"], s=5, color="0.8", lw=0, alpha=0.5, rasterized=True)
        sc = ax.scatter(g.loc[an, "age"], g.loc[an, "lfd"], c=g.loc[an, "ldg"], s=19, cmap="viridis", vmin=VMIN, vmax=VMAX, edgecolor="0.2", lw=0.3)
        ax.set_title(f"{BOX_STYLE[b]['label']}: {int(an.sum())} analogues", fontsize=9.5)
    ax.set_xlabel("age [Gyr]"); ax.grid(alpha=0.25, lw=0.5)
np.atleast_1d(axs)[0].set_ylabel(r"$\log M_{\rm dust}/M_\star$")
cb = fig.colorbar(sc, ax=np.atleast_1d(axs).tolist(), pad=0.012, fraction=0.03)
cb.set_label(r"$\log M_{\rm dust}/M_{\rm gas}$")
paper_save(fig, "noxray_analogues_dgr_plane")
plt.show()

## Part 3 — the AGN history of the nox analogues: weak coupling, or no exposure?

s50nox itself has no histories (its catalogue ladder is not on the share), but the *analogues* live in boxes whose histories are built, so their pre- and post-quenching AGN record exists: $w_{\rm pre}$ and the weak / strong class (Part 6a of the boxes notebook), the first gated X-ray episode's timing (`agn_onset`: lead / concurrent / late / never) and the post-SFT exposure integrals (Part 9: $E_x$, $f_x$, $r_x$, $T_{\rm gas\text{-}poor}$, the gate's lag, the quenching sequence). Per analogue box, analogues against the rest of the same quenched sample:

* **(a) the coupling** — the weak / strong mix (Fisher), the $w_{\rm pre}$ distributions (Mann–Whitney, AUC): *is the analogue population weakly coupled?*
* **(b) the exposure** — $E_x$, $f_x$, $T_{\rm gas\text{-}poor}$, the gate's lag, the never-exposed fraction ($E_x = 0$), the onset mix: *or did the channel simply never (or barely, or only lately) fire on them?*
* **(c) the BH and the environment** — $\log M_{\rm BH}$, the jet flag and the undergrown-BH flag at the anchor, the central / satellite mix, $M_\star$, $z$: the boxes notebook found the never-exposed fiducial QGs are undergrown-BH satellites — are the analogues those same galaxies? (cross-tables analogue $\times$ never-exposed and analogue $\times$ BH-not-grown).

All contrasts $\to$ `noxray_analogues_agnhistory.csv`; figure `noxray_analogues_agnhistory`: per box the $w_{\rm pre}$ and $f_x$ CDFs (analogues vs the rest, the class cuts marked, the never-exposed fractions and the median $E_x$ annotated) and the onset mix. A *chain check* prints the fraction whose anchor BH is below the jet threshold while $E_x > 0$ — a satellite's most-massive-progenitor chain can run through its central, inheriting the bigger system's $M_{\rm BH}$ / $f_{\rm Edd}$ / $f_{\rm gas}$, so satellite $w_{\rm pre}$ / $E_x$ are upper limits — and every exposure contrast is repeated on the centrals and the satellites separately. The verdict line at the end states which of the two readings (weak coupling / no exposure) the numbers support.

In [ ]:
# ── Part 3 — the AGN history of the nox analogues: weak coupling, or no exposure? ──
QTY3 = [("w_pre", "w_pre (pre-SFT jet weight)"), ("e_x", "E_x [Gyr]"), ("f_x", "f_x (duty since SFT)"), ("r_x", "r_x (jet wt while gas-poor)"),
        ("t_gaspoor", "T_gaspoor [Gyr]"), ("lag_gate", "t_gate - t_SFT [Gyr]"), ("dt_sft", "t_anchor - t_SFT [Gyr]"), ("tau_q", "tau_q [Gyr]"),
        ("log_mbh", "log M_BH (anchor)"), ("log_mstar", "log M*"), ("age", "age [Gyr]")]
ONSETS = [("lead", "#fdae61"), ("concurrent", "#d7191c"), ("late", "#2c7bb6"), ("never", "0.35")]   # first gated X-ray episode vs SFT / QT (boxes Part 6a)
MIN_AN = 5
rows3 = []

fig, axs = plt.subplots(len(ANALOG_BOXES), 3, figsize=(12.8, 3.9 * len(ANALOG_BOXES)), squeeze=False)
for i, b in enumerate(ANALOG_BOXES):
    g = QM[QM["box"] == b]; an = g["nox_analog"].to_numpy(bool); c = BOX_STYLE[b]["color"]
    hist = np.isfinite(g["e_x"].to_numpy(float))
    print(f"\n{'=' * 100}\n{b}: {int(an.sum())} analogues; history coverage (finite E_x): {100 * hist[an].mean():.0f} % of the analogues, {100 * hist[~an].mean():.0f} % of the rest"
          + (" (only the z >= 1.15 anchors of cis50 have histories)" if b == "cis50" else ""))
    if an.sum() < MIN_AN:
        print(f"  fewer than {MIN_AN} analogues: contrasts skipped")
        for j in range(3):
            axs[i, j].axis("off")
        axs[i, 0].set_title(f"{BOX_STYLE[b]['label']}: {int(an.sum())} analogues — too few", loc="left", fontsize=10.5)
        continue

    # (a) the coupling: the class mix and w_pre
    gc = g[g["agn_class"].isin(CLASSES)]; anc = gc["nox_analog"].to_numpy(bool)
    wa, sa = int(((gc["agn_class"] == "weak") & anc).sum()), int(((gc["agn_class"] == "strong") & anc).sum())
    wr, sr = int(((gc["agn_class"] == "weak") & ~anc).sum()), int(((gc["agn_class"] == "strong") & ~anc).sum())
    p_cls = fisher_exact([[wa, sa], [wr, sr]])[1] if min(wa + sa, wr + sr) else np.nan
    print(f"  (a) coupling class:  analogues weak {wa} / strong {sa} ({100 * wa / max(wa + sa, 1):.0f} % weak)  vs  rest weak {wr} / strong {sr} "
          f"({100 * wr / max(wr + sr, 1):.0f} % weak); Fisher p = {p_cls:.2g}")

    # (b) the exposure: never-exposed fraction and the onset mix
    ge = g[np.isfinite(g["e_x"])]; ane = ge["nox_analog"].to_numpy(bool); nev = (ge["e_x"] <= 0).to_numpy()
    p_nev = fisher_exact([[int((nev & ane).sum()), int((~nev & ane).sum())], [int((nev & ~ane).sum()), int((~nev & ~ane).sum())]])[1] if min(ane.sum(), (~ane).sum()) else np.nan
    print(f"  (b) never exposed (E_x = 0): analogues {100 * nev[ane].mean():.0f} % ({int(nev[ane].sum())}/{int(ane.sum())})  vs  rest {100 * nev[~ane].mean():.0f} % "
          f"({int(nev[~ane].sum())}/{int((~ane).sum())}); Fisher p = {p_nev:.2g}")
    go = g[g["agn_onset"].isin([o for o, _ in ONSETS])]; ano = go["nox_analog"].to_numpy(bool)
    for name, mm in [("analogues", ano), ("rest", ~ano)]:
        h = go[mm]
        print(f"      onset mix, {name:9s} (N = {len(h):4d}): " + ", ".join(f"{o} {100 * (h['agn_onset'] == o).mean():.0f} %" for o, _ in ONSETS))

    # (c) the BH and the environment: undergrown BH, satellites, and every quantity's contrast
    lmbh = g["log_mbh"].to_numpy(float)
    low = ~np.isfinite(lmbh) | (lmbh <= JET_LOGMBH)
    p_low = fisher_exact([[int(low[an].sum()), int((~low)[an].sum())], [int(low[~an].sum()), int((~low)[~an].sum())]])[1]
    print(f"  (c) BH not grown (log M_BH <= {JET_LOGMBH:g} or none): analogues {100 * low[an].mean():.0f} %  vs  rest {100 * low[~an].mean():.0f} % (Fisher p = {p_low:.2g}); "
          f"satellites: analogues {100 * (1 - g.loc[an, 'central'].mean()):.0f} % vs rest {100 * (1 - g.loc[~an, 'central'].mean()):.0f} %; "
          f"jet at the anchor: {100 * g.loc[an, 'jet'].mean():.0f} % vs {100 * g.loc[~an, 'jet'].mean():.0f} %")
    print(f"      {'quantity':28s} {'analog med':>10s} {'rest med':>9s} {'delta':>7s} {'p':>9s} {'AUC':>5s}   (AUC = P(analogue > rest))")
    for col, lab in QTY3:
        auc, p, na_, nr_ = mw_auc(g[col], an)
        ma, mr = med3(g.loc[an, col])[0], med3(g.loc[~an, col])[0]
        rows3.append(dict(box=b, quantity=col, label=lab, n_analog=na_, n_rest=nr_, med_analog=ma, med_rest=mr, delta=ma - mr, p=p, auc=auc))
        r = rows3[-1]
        print(f"      {lab:28s} {r['med_analog']:10.2f} {r['med_rest']:9.2f} {r['delta']:+7.2f} {r['p']:9.2g} {r['auc']:5.2f}")
    rows3.append(dict(box=b, quantity="frac_weak", label="weak fraction (classified)", n_analog=wa + sa, n_rest=wr + sr,
                      med_analog=wa / max(wa + sa, 1), med_rest=wr / max(wr + sr, 1), delta=wa / max(wa + sa, 1) - wr / max(wr + sr, 1), p=p_cls, auc=np.nan))
    rows3.append(dict(box=b, quantity="frac_never", label="never exposed (E_x = 0)", n_analog=int(ane.sum()), n_rest=int((~ane).sum()),
                      med_analog=float(nev[ane].mean()), med_rest=float(nev[~ane].mean()), delta=float(nev[ane].mean() - nev[~ane].mean()), p=p_nev, auc=np.nan))
    rows3.append(dict(box=b, quantity="frac_lowbh", label="BH not grown at the anchor", n_analog=int(an.sum()), n_rest=int((~an).sum()),
                      med_analog=float(low[an].mean()), med_rest=float(low[~an].mean()), delta=float(low[an].mean() - low[~an].mean()), p=p_low, auc=np.nan))

    # the chain caveat and the centrals / satellites robustness: a satellite's most-massive-progenitor chain can run through its central,
    # inheriting the bigger system's M_BH / f_Edd / f_gas — its w_pre and E_x are then upper limits, while the anchor BH state is chain-independent
    inh = low & (g["e_x"].to_numpy(float) > 0)
    print(f"  chain check: BH below the jet threshold at the anchor yet E_x > 0 — analogues {100 * inh[an].mean():.0f} %, rest {100 * inh[~an].mean():.0f} %: "
          f"their w_pre / E_x are inherited along the chain (satellite exposures = upper limits)")
    cen = g["central"].to_numpy(float) > 0
    for sub, mm in [("centrals", cen), ("satellites", ~cen)]:
        hh = g[mm]; anh = hh["nox_analog"].to_numpy(bool)
        if anh.sum() < MIN_AN:
            print(f"      {sub} only: {int(anh.sum())} analogues — too few for the contrast"); continue
        fine = np.isfinite(hh["e_x"].to_numpy(float)); nevh = (hh["e_x"].to_numpy(float) <= 0) & fine
        s = f"      {sub} only ({int(anh.sum())} an.): never exp {100 * nevh[anh].sum() / max(fine[anh].sum(), 1):.0f} % vs {100 * nevh[~anh].sum() / max(fine[~anh].sum(), 1):.0f} %"
        for col in ("w_pre", "e_x", "f_x", "r_x"):
            auc_, _, _, _ = mw_auc(hh[col], anh)
            s += f"; {col} {med3(hh.loc[anh, col])[0]:.2f} vs {med3(hh.loc[~anh, col])[0]:.2f} (AUC {auc_:.2f})"
        print(s)

    # the verdict line: which reading separates the analogues more — the coupling (w_pre) or the exposure (E_x / f_x / r_x)?
    _auc = {q: next(r for r in rows3 if r["box"] == b and r["quantity"] == q)["auc"] for q in ("w_pre", "e_x", "f_x", "r_x")}
    _sep = {q: abs(a - 0.5) for q, a in _auc.items() if np.isfinite(a)}
    _sw = _sep.get("w_pre", np.nan)
    _qx = max((q for q in ("e_x", "f_x", "r_x") if q in _sep), key=lambda q: _sep[q], default=None)
    if _qx is None or not np.isfinite(_sw):
        print(f"  verdict {b}: too few analogues with histories for a verdict" + (" — separations |AUC-0.5|: " + ", ".join(f"{q} {v:.2f}" for q, v in _sep.items()) if _sep else ""))
    else:
        lean = f"the EXPOSURE ({_qx}, |AUC-0.5| = {_sep[_qx]:.2f})" if _sep[_qx] > _sw else f"the COUPLING (w_pre, |AUC-0.5| = {_sw:.2f})"
        print(f"  verdict {b}: |AUC-0.5| w_pre {_sw:.2f} vs exposure e_x {_sep.get('e_x', np.nan):.2f} / f_x {_sep.get('f_x', np.nan):.2f} / r_x {_sep.get('r_x', np.nan):.2f}"
              f" -> the stronger separator of the nox-like phenotype is {lean}")

    # the figure row
    r_w = next(r for r in rows3 if r["box"] == b and r["quantity"] == "w_pre")
    r_e = next(r for r in rows3 if r["box"] == b and r["quantity"] == "e_x")
    r_f = next(r for r in rows3 if r["box"] == b and r["quantity"] == "f_x")
    ax = axs[i, 0]
    ecdf(ax, g.loc[~an, "w_pre"], color="0.6", lw=1.8, label="rest")
    ecdf(ax, g.loc[an, "w_pre"], color=c, lw=2.8, label="analogues")
    for x0, ls in [(PRE_THR_WEAK, ":"), (TWO_CLASS_THR, "--"), (PRE_THR_STRONG, ":")]:
        ax.axvline(x0, color="0.3", ls=ls, lw=1.0)
    ax.set_xlabel(r"$w_{\rm pre}$ (pre-SFT jet weight; cuts 0.10 / 0.30 / 0.50)"); ax.set_ylabel("CDF"); ax.set_ylim(0, 1); ax.grid(alpha=0.25, lw=0.5)
    ax.text(0.97, 0.06, f"p = {r_w['p']:.1g}, AUC = {r_w['auc']:.2f}", transform=ax.transAxes, ha="right", fontsize=8.5)
    ax.set_title(f"{BOX_STYLE[b]['label']}: {int(an.sum())} analogues", loc="left", fontsize=10.5)
    ax = axs[i, 1]
    ecdf(ax, g.loc[~an, "f_x"], color="0.6", lw=1.8)
    ecdf(ax, g.loc[an, "f_x"], color=c, lw=2.8)
    ax.set_xlabel(r"$f_x = E_x / (t_{\rm anchor} - t_{\rm SFT})$ (X-ray duty since SFT)"); ax.set_ylim(0, 1); ax.grid(alpha=0.25, lw=0.5)
    ax.text(0.03, 0.96, f"p = {r_f['p']:.1g}, AUC = {r_f['auc']:.2f}\nnever exposed: {100 * nev[ane].mean():.0f} % vs {100 * nev[~ane].mean():.0f} %\n"
            f"median $E_x$: {r_e['med_analog']:.2f} vs {r_e['med_rest']:.2f} Gyr", transform=ax.transAxes, va="top", fontsize=8.5)
    ax = axs[i, 2]
    for yi, (name, mm) in enumerate([("analogues", ano), ("rest", ~ano)]):
        h = go[mm]; left = 0.0
        for o, ccol in ONSETS:
            fr = float((h["agn_onset"] == o).mean()) if len(h) else 0.0
            ax.barh(yi, fr, left=left, color=ccol, edgecolor="white", height=0.55)
            if fr > 0.07:
                ax.text(left + fr / 2, yi, f"{100 * fr:.0f}%", ha="center", va="center", fontsize=8, color="0.1" if o == "lead" else "white")
            left += fr
    ax.set_yticks([0, 1]); ax.set_yticklabels(["analogues", "rest"]); ax.set_xlim(0, 1); ax.set_ylim(-0.6, 1.6)
    ax.set_xlabel("first gated X-ray episode")
    if i == 0:
        ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=ccol, label=o) for o, ccol in ONSETS], loc="upper center", ncol=4, fontsize=7.6, frameon=False, bbox_to_anchor=(0.5, 1.14))
        axs[0, 0].legend(loc="center right", fontsize=8)

HIST_CSV = os.path.join(OUT, "noxray_analogues_agnhistory.csv")
pd.DataFrame(rows3).to_csv(HIST_CSV, index=False)
print(f"\ncontrasts -> {HIST_CSV}")
fig.tight_layout()
paper_save(fig, "noxray_analogues_agnhistory")
plt.show()

## Part 4 — selecting dusty quenched galaxies in real life: $\kappa_{\rm rot}$ gas against $\Sigma_{\rm e}$

If the dusty quenched galaxies of the AGN-on universe are the nox-like, gas-rotating, low-concentration systems, then two quantities an observer can actually reach — the cold-gas rotation (CO / [CII] moment maps; $\kappa_{\rm rot}$ gas is the simulation's proxy for a rotation-dominated velocity field) and the stellar surface density $\Sigma_{\rm e}$ (imaging) — should pick them out *without ever measuring the dust*. Per AGN-on box, on the whole quenched sample:

* the **AUC of every structural / cheap quantity** ($\kappa_{\rm gas}$, $\kappa_\star$, $\Sigma_{\rm e}$, $R_{\rm e}$, $\Delta \log R_{\rm e}$, age, $M_\star$, sSFR) for dust-rich membership ($\log M_{\rm dust}/M_\star \geq$ `DUSTY_CUT`, the ALMA-C11 locus);
* the **best two-cut rule** $\kappa_{\rm gas} \geq a$ and $\log \Sigma_{\rm e} \leq s$ (grid-searched, F1-optimal) with its completeness and purity against the base rate, plus the one-cut $\kappa_{\rm gas} \geq$ `DISC_KAPPA` rule for reference $\to$ `noxray_analogues_structure.csv`;
* the figure `noxray_analogues_structure`: the ($\Sigma_{\rm e}$, $\kappa_{\rm gas}$) plane per box coloured by $\log M_{\rm dust}/M_\star$, the best rule drawn as the two cuts, s50nox as the last panel (where essentially everything is dusty — the plane's dusty corner is the nox locus).

The caveat travels with the result: $\kappa_{\rm rot}$ of the *member gas* is not a direct observable, and in the coarse boxes the quenched gas reservoirs are a few dozen particles — treat the numbers as a direction (rotating cold gas + low $\Sigma_{\rm e}$), not a calibrated selection function.

In [ ]:
# ── Part 4 — the structural selection of dusty quenched galaxies: AUCs, the kappa_gas x Sigma_e rule, the plane ──
DUSTY_CUT = -3.5     # dust-rich: log M_dust/M* at or above this (the ALMA-C11 detections; the boxes notebook's P8 edge)
S4_PRED = [("kappa_gas", "kappa_rot gas"), ("kappa_star", "kappa_rot stars"), ("lsig", "log Sigma_e"), ("re", "R_e [kpc]"),
           ("dlre", "dlog R_e | M*,z"), ("age", "age [Gyr]"), ("log_mstar", "log M*"), ("lssfr", "log sSFR")]
rows4, best_rule = [], {}
for b in ANALOG_BOXES:
    g = QM[QM["box"] == b]
    dusty = (g["lfd"].to_numpy(float) >= DUSTY_CUT)
    base = dusty.mean()
    print(f"\n{b}: dust-rich = log M_dust/M* >= {DUSTY_CUT:g}: {int(dusty.sum())} of {len(g)} quenched ({100 * base:.0f} % base rate)")
    print(f"    {'predictor':18s} {'AUC':>5s} {'p':>9s}   (AUC = P(dust-rich > rest); < 0.5 = lower values pick the dusty)")
    for col, lab in S4_PRED:
        auc, p, _, _ = mw_auc(g[col], dusty)
        rows4.append(dict(box=b, kind="auc", rule=col, label=lab, value=auc, p=p, completeness=np.nan, purity=np.nan, n_sel=np.nan, n_dusty=int(dusty.sum()), n=len(g)))
        print(f"    {lab:18s} {auc:5.2f} {p:9.2g}")
    kg, ls_ = g["kappa_gas"].to_numpy(float), g["lsig"].to_numpy(float)
    fin = np.isfinite(kg) & np.isfinite(ls_)

    def score(sel):
        comp = (sel & dusty).sum() / max(dusty[fin].sum(), 1)
        pur = (sel & dusty).sum() / max(sel.sum(), 1)
        return comp, pur, 2 * comp * pur / max(comp + pur, 1e-9)

    best = None
    for a in np.arange(0.05, 0.96, 0.05):
        for s in np.nanpercentile(ls_[fin], np.arange(10, 91, 5)):
            sel = fin & (kg >= a) & (ls_ <= s)
            if sel.sum() < 5:
                continue
            comp, pur, f1 = score(sel)
            if best is None or f1 > best[0]:
                best = (f1, a, s, comp, pur, int(sel.sum()))
    if best:
        f1, a, s, comp, pur, nsel = best
        best_rule[b] = (a, s)
        rows4.append(dict(box=b, kind="rule", rule=f"kappa_gas>={a:.2f} & lsig<={s:.2f}", label="best F1 two-cut rule", value=f1, p=np.nan,
                          completeness=comp, purity=pur, n_sel=nsel, n_dusty=int(dusty.sum()), n=len(g)))
        print(f"    best rule: kappa_gas >= {a:.2f} and log Sigma_e <= {s:.2f} -> completeness {100 * comp:.0f} %, purity {100 * pur:.0f} % "
              f"(base rate {100 * base:.0f} %), F1 {f1:.2f}, N selected {nsel}")
    sel1 = fin & (kg >= DISC_KAPPA)
    comp, pur, f1 = score(sel1)
    rows4.append(dict(box=b, kind="rule", rule=f"kappa_gas>={DISC_KAPPA:g}", label="one-cut gas-disc rule", value=f1, p=np.nan,
                      completeness=comp, purity=pur, n_sel=int(sel1.sum()), n_dusty=int(dusty.sum()), n=len(g)))
    print(f"    one-cut rule kappa_gas >= {DISC_KAPPA:g}          -> completeness {100 * comp:.0f} %, purity {100 * pur:.0f} %, F1 {f1:.2f}, N selected {int(sel1.sum())}")
STRUCT_CSV = os.path.join(OUT, "noxray_analogues_structure.csv")
pd.DataFrame(rows4).to_csv(STRUCT_CSV, index=False)
print(f"\nrules / AUCs -> {STRUCT_CSV}")

# ── the plane: kappa_gas vs log Sigma_e coloured by log M_dust/M*, the best rule drawn; s50nox as the reference panel ──
_pb = ANALOG_BOXES + [NOX_BOX]
fig, axs = plt.subplots(1, len(_pb), figsize=(3.9 * len(_pb), 3.9), sharex=True, sharey=True)
for ax, b in zip(np.atleast_1d(axs), _pb):
    g = QM[QM["box"] == b]
    sc = ax.scatter(g["lsig"], g["kappa_gas"], c=g["lfd"].clip(lower=P9_FLOOR), s=9, cmap="magma", vmin=P9_FLOOR, vmax=-2.6, lw=0, rasterized=True)
    if b in best_rule:
        a, s = best_rule[b]
        ax.axhline(a, color="#009E73" if False else "0.15", ls="--", lw=1.2)
        ax.axvline(s, color="0.15", ls="--", lw=1.2)
        ax.set_title(f"{BOX_STYLE[b]['label']}\n" + rf"rule: $\kappa_{{\rm gas}} \geq {a:.2f}$, $\log \Sigma_{{\rm e}} \leq {s:.2f}$", fontsize=9)
    else:
        ax.set_title(f"{BOX_STYLE[b]['label']}\n(no X-ray: the dusty corner's origin)", fontsize=9)
    ax.set_xlabel(r"$\log \Sigma_{\rm e}$ [$M_\odot$ kpc$^{-2}$]"); ax.grid(alpha=0.25, lw=0.5)
np.atleast_1d(axs)[0].set_ylabel(r"$\kappa_{\rm rot}$ gas")
cb = fig.colorbar(sc, ax=np.atleast_1d(axs).tolist(), pad=0.012, fraction=0.03)
cb.set_label(r"$\log M_{\rm dust}/M_\star$")
paper_save(fig, "noxray_analogues_structure")
plt.show()

## Part 5 — why does the structural selection work only in m25: resolution, statistics, or the BH?

Part 4 left $\kappa_{\rm gas}$ an AUC-0.69 predictor of dust-rich membership in m25 and a 0.53–0.55 one in m50 / m100. Three explanations compete, and each leaves a different catalogue-level fingerprint:

* **(a) low statistics of the 25 Mpc box** — then a m25-*sized* draw from m100 should reach AUC 0.69 reasonably often. Test: bootstrap 16–84 % intervals per box, plus `P5_NDRAW` pseudo-m25 samples drawn from m100 (per anchor, each m25 galaxy replaced by a $|\Delta \log M_\star| \leq 0.2$ twin) $\to$ the null distribution of the AUC and $P({\rm AUC} \geq {\rm m25's})$.
* **(b) the gas-mass composition and the selection floor** — the $\geq 21$-particle cut keeps reservoirs down to $4.8\times10^7 M_\odot$ in m25 but only $3.8\times10^8$ in m100 / m50, and within a window of **matched $M_{\rm gas}$** the same reservoir holds $8\times$ the particles in m25. If the AUCs converge at matched $M_{\rm gas}$, the box difference is *which* reservoirs each sample holds, not how well $\kappa$ is measured.
* **(c) the BH axis** — m25 grows smaller BHs. If that made the signal, it should vanish within $M_{\rm BH}$ strata. Test: the AUC inside $\log M_{\rm BH} \leq 7.5$ / 7.5–8.5 / $> 8.5$ per box, and the $N$-weighted stratum average.

Also printed: the $\kappa_{\rm gas}$ ngas terciles per box (confounded with $M_{\rm gas}$ within a box — shown for completeness) and the raw separation (median $\kappa_{\rm gas}$ of dusty vs the rest — m25's is *positive*, the coarse boxes' is *negative*: a sign flip that pure measurement noise cannot produce). Everything $\to$ `noxray_analogues_resolution_tests.{png,pdf,csv}`.

In [ ]:
# ── Part 5 — resolution, statistics, or the BH? the catalogue-level discriminating tests ──
from scipy.stats import rankdata
P5_NDRAW, P5_NBOOT = 2000, 2000
P5_TWIN_DM = 0.2                     # a pseudo-m25 twin must match the m25 galaxy's anchor and log M* within this
P5_MGAS_WIN = (21 * 1.82e7, 3e9)     # matched-M_gas window: the m100 gas-mass floor to 3e9 Msun
RNG5 = np.random.default_rng(7)
P5_BOXES = ["cis50", "cis100", "cis25"]


def auc_rank(x, y):
    """AUC = P(x | y=True  >  x | y=False) via ranks (fast enough for the bootstrap)."""
    x = np.asarray(x, float); y = np.asarray(y, bool)
    fin = np.isfinite(x); x, y = x[fin], y[fin]
    n1, n0 = int(y.sum()), int((~y).sum())
    if n1 < 3 or n0 < 3:
        return np.nan
    r = rankdata(x)
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


B5 = {b: QM[QM["box"] == b].copy() for b in P5_BOXES}
for g in B5.values():
    g["dusty"] = g["lfd"] >= DUSTY_CUT
rows5 = []

# (a) the AUC with its bootstrap interval, and the pseudo-m25 null from m100
print(f"(a) AUC of kappa_gas for dust-rich (log f_dust >= {DUSTY_CUT:g}), bootstrap 16-84 %:")
CI5 = {}
for b, g in B5.items():
    x, y = g["kappa_gas"].to_numpy(float), g["dusty"].to_numpy(bool)
    fin = np.isfinite(x); x, y = x[fin], y[fin]
    bs = [auc_rank(x[i], y[i]) for i in (RNG5.integers(0, len(x), (P5_NBOOT, len(x))))]
    a, (lo, hi) = auc_rank(x, y), np.percentile([v for v in bs if np.isfinite(v)], [16, 84])
    CI5[b] = (a, lo, hi)
    rows5.append(dict(kind="auc_boot", box=b, value=a, lo=lo, hi=hi, n=len(g), n_dusty=int(g["dusty"].sum()), note=""))
    print(f"  {b:7s} N={len(g):5d} dusty={int(g['dusty'].sum()):4d}  AUC {a:.3f}  ({lo:.3f}-{hi:.3f})")
g25, g100 = B5["cis25"], B5["cis100"]
pools = [p for _, r in g25.iterrows()
         if len(p := g100.index[(g100["snap"] == r["snap"]).to_numpy() & (np.abs(g100["log_mstar"] - r["log_mstar"]) <= P5_TWIN_DM).to_numpy()].to_numpy())]
PSEUDO = []
for _ in range(P5_NDRAW):
    h = g100.loc[[RNG5.choice(p) for p in pools]]
    PSEUDO.append(auc_rank(h["kappa_gas"], h["dusty"]))
PSEUDO = np.array([v for v in PSEUDO if np.isfinite(v)])
p_stat = float(np.mean(PSEUDO >= CI5["cis25"][0]))
rows5.append(dict(kind="pseudo_m25", box="cis100", value=float(np.median(PSEUDO)), lo=float(np.percentile(PSEUDO, 16)), hi=float(np.percentile(PSEUDO, 84)),
                  n=len(pools), n_dusty=np.nan, note=f"P(AUC >= m25's) = {p_stat:.4f}"))
print(f"  pseudo-m25 from m100 ({P5_NDRAW} draws, twins for {len(pools)}/{len(g25)} galaxies): AUC {np.median(PSEUDO):.3f} "
      f"({np.percentile(PSEUDO, 16):.3f}-{np.percentile(PSEUDO, 84):.3f}); P(>= m25's {CI5['cis25'][0]:.3f}) = {p_stat:.4f} -> statistics "
      + ("CANNOT" if p_stat < 0.05 else "could") + " explain the m25 signal")

# (b) matched M_gas across the boxes, and the m25 galaxies below the m100 floor
print(f"\n(b) matched M_gas [{P5_MGAS_WIN[0]:.1e}, {P5_MGAS_WIN[1]:.1e}) — the same reservoirs, 8x the particles in m25:")
for b, g in B5.items():
    h = g[(g["mgas"] >= P5_MGAS_WIN[0]) & (g["mgas"] < P5_MGAS_WIN[1])]
    a = auc_rank(h["kappa_gas"], h["dusty"])
    rows5.append(dict(kind="mgas_window", box=b, value=a, lo=np.nan, hi=np.nan, n=len(h), n_dusty=int(h["dusty"].sum()), note=f"med ngas {h['ngas'].median():.0f}"))
    print(f"  {b:7s} N={len(h):5d} dusty={int(h['dusty'].sum()):4d}  AUC {a:.3f}  med ngas {h['ngas'].median():.0f}")
sub = g25[g25["mgas"] < P5_MGAS_WIN[0]]
print(f"  m25 BELOW the m100 floor (reservoirs m100 cannot even keep): N={len(sub)}, dusty={int(sub['dusty'].sum())}")
rows5.append(dict(kind="below_floor", box="cis25", value=np.nan, lo=np.nan, hi=np.nan, n=len(sub), n_dusty=int(sub["dusty"].sum()), note="m25 galaxies under the m100 gas-mass floor"))

# (c) the BH axis: the AUC inside M_BH strata
print(f"\n(c) the AUC inside M_BH strata (if the BH mix made the signal, it would vanish at fixed M_BH):")
STRATA5 = [("BH<=7.5", lambda lm: ~np.isfinite(lm) | (lm <= 7.5)), ("7.5-8.5", lambda lm: np.isfinite(lm) & (lm > 7.5) & (lm <= 8.5)), (">8.5", lambda lm: np.isfinite(lm) & (lm > 8.5))]
for b, g in B5.items():
    lm = g["log_mbh"].to_numpy(float); ws, vs, cells = [], [], []
    for name, fm in STRATA5:
        h = g[fm(lm)]; a = auc_rank(h["kappa_gas"], h["dusty"])
        rows5.append(dict(kind="mbh_stratum", box=b, value=a, lo=np.nan, hi=np.nan, n=len(h), n_dusty=int(h["dusty"].sum()), note=name))
        cells.append(f"{name}: {a:.2f} (N {len(h)}, dusty {100 * h['dusty'].mean():.0f} %)")
        if np.isfinite(a):
            ws.append(len(h)); vs.append(a)
    wa = float(np.average(vs, weights=ws)) if vs else np.nan
    rows5.append(dict(kind="mbh_weighted", box=b, value=wa, lo=np.nan, hi=np.nan, n=int(np.sum(ws)), n_dusty=int(g["dusty"].sum()), note="N-weighted over the strata"))
    print(f"  {b:7s} " + " | ".join(cells) + f"  -> at fixed M_BH: {wa:.2f}")

# (d) ngas terciles within each box (confounded with M_gas within a box) and the raw separation (the sign flip)
print(f"\n(d) ngas terciles within each box (ngas ~ M_gas at fixed resolution: physics and sampling mixed — completeness only):")
for b, g in B5.items():
    qs = np.nanpercentile(g["ngas"], [33, 66])
    cells = []
    for h in (g[g["ngas"] <= qs[0]], g[(g["ngas"] > qs[0]) & (g["ngas"] <= qs[1])], g[g["ngas"] > qs[1]]):
        a = auc_rank(h["kappa_gas"], h["dusty"]); cells.append(f"{a:.2f} (N {len(h)})")
        rows5.append(dict(kind="ngas_tercile", box=b, value=a, lo=np.nan, hi=np.nan, n=len(h), n_dusty=int(h["dusty"].sum()), note=f"edges {qs[0]:.0f}/{qs[1]:.0f}"))
    print(f"  {b:7s} AUC " + " / ".join(cells))
print(f"\n(e) the separation itself (median kappa_gas, dusty vs the rest — the m25 sign flip pure noise cannot produce):")
for b, g in B5.items():
    d, r = g.loc[g["dusty"], "kappa_gas"].dropna(), g.loc[~g["dusty"], "kappa_gas"].dropna()
    rows5.append(dict(kind="separation", box=b, value=float(d.median() - r.median()), lo=float(d.median()), hi=float(r.median()), n=len(g), n_dusty=len(d), note="value = dusty - rest median kappa_gas"))
    print(f"  {b:7s} dusty {d.median():.2f} vs rest {r.median():.2f} (delta {d.median() - r.median():+.2f})")
RES_CSV = os.path.join(OUT, "noxray_analogues_resolution_tests.csv")
pd.DataFrame(rows5).to_csv(RES_CSV, index=False)
print(f"tests -> {RES_CSV}")

# ── the figure: the pseudo-m25 null, the matched-M_gas convergence, the M_BH strata ──
fig, axs = plt.subplots(1, 3, figsize=(13.0, 3.8))
ax = axs[0]
ax.hist(PSEUDO, bins=36, color="0.75", density=True, label=f"m100 drawn as m25 ({P5_NDRAW}x)")
for b in P5_BOXES:
    a, lo, hi = CI5[b]
    ax.axvline(a, color=BOX_STYLE[b]["color"], lw=2.2)
    ax.axvspan(lo, hi, color=BOX_STYLE[b]["color"], alpha=0.12)
ax.set_xlabel(r"AUC of $\kappa_{\rm gas}$ for dust-rich"); ax.set_ylabel("density")
ax.set_title(f"(a) statistics: P(pseudo-m25 $\\geq$ m25) = {p_stat:.3f}", fontsize=10)
ax.legend(fontsize=7.6, loc="upper left")
ax = axs[1]
for j, b in enumerate(P5_BOXES):
    g = B5[b]; h = g[(g["mgas"] >= P5_MGAS_WIN[0]) & (g["mgas"] < P5_MGAS_WIN[1])]
    ax.plot(j - 0.12, CI5[b][0], "o", color=BOX_STYLE[b]["color"], ms=9)
    ax.plot([j - 0.12] * 2, CI5[b][1:3], "-", color=BOX_STYLE[b]["color"], lw=1.6)
    ax.plot(j + 0.12, auc_rank(h["kappa_gas"], h["dusty"]), "s", color=BOX_STYLE[b]["color"], ms=9, mfc="none", mew=2)
ax.axhline(0.5, color="0.4", ls=":", lw=1)
ax.set_xticks(range(len(P5_BOXES))); ax.set_xticklabels([b.replace("cis", "m") for b in P5_BOXES])
ax.set_ylabel("AUC"); ax.set_title(r"(b) full sample (filled) vs matched $M_{\rm gas}$ (open)", fontsize=10)
ax = axs[2]
for j, b in enumerate(P5_BOXES):
    lm = B5[b]["log_mbh"].to_numpy(float)
    for k, (name, fm) in enumerate(STRATA5):
        h = B5[b][fm(lm)]; a = auc_rank(h["kappa_gas"], h["dusty"])
        if np.isfinite(a):
            ax.plot(j + (k - 1) * 0.22, a, "o^s"[k], color=BOX_STYLE[b]["color"], ms=8, mfc="none" if k != 1 else BOX_STYLE[b]["color"], mew=1.6)
ax.axhline(0.5, color="0.4", ls=":", lw=1)
ax.set_xticks(range(len(P5_BOXES))); ax.set_xticklabels([b.replace("cis", "m") for b in P5_BOXES])
ax.set_ylabel("AUC"); ax.set_title(r"(c) inside $M_{\rm BH}$ strata (o $\leq 7.5$ / ^ 7.5-8.5 / s $> 8.5$)", fontsize=10)
for ax in axs:
    ax.grid(alpha=0.25, lw=0.5)
fig.tight_layout()
paper_save(fig, "noxray_analogues_resolution_tests")
plt.show()

## Part 5b — the degradation experiment: measure $\kappa_{\rm gas}$ of the m25 galaxies at m100 sampling

Part 5 excluded statistics and the BH mix, and showed the matched-$M_{\rm gas}$ composition carries part of the box difference. What remains untested is the **estimator channel**: is $\kappa_{\rm rot}$ from a $\sim$50-particle reservoir simply too noisy to rank the dusty galaxies? This cell measures exactly that, on the *same galaxies*:

1. for every cis25 quenched galaxy of the sample, the **member gas** (caesar `glist`) is read from the m25 reduced particle files (`output/cis25/reduced_particles/`, built by `build_reduced_particles_job.py`: `pos` kpc relative to the centre, `vel` peculiar km/s, `member` = the caesar particle list — so this is the same gas caesar used);
2. $\kappa_{\rm rot}$ is recomputed with the Sales+10 definition (ordered-rotation kinetic energy fraction about the member gas' own angular momentum, COM velocity frame) at **full sampling** — the check against the catalogue `gas_kappa_rot` is printed (this validates the implementation);
3. each galaxy is then **resampled to the particle count m100 would give its reservoir**, $n_{100} = M_{\rm gas}^{\rm member} / 1.82\times10^7$ (galaxies with $n_{100} < 21$ are flagged: they would not even pass the m100 selection), `P5B_NREAL` random draws;
4. the AUC of the degraded $\kappa$ for dust-rich membership is measured per realisation — with the under-floor galaxies kept and with them dropped (the honest m100 selection).

**Reading it**: if the degraded AUC falls to the m100 value ($\sim$0.55), the coarse boxes fail because the estimator is noise-limited at their sampling — and $\kappa_{\rm gas}$ remains a *physically* valid selector that m50 / m100 simply cannot measure. If it stays near the full-sampling value, the estimator is fine even at 50 particles, and the coarse boxes' failure is physical (their dusty QGs genuinely are not rotating discs — the Part 2b gas-rich passers-through), i.e. a resolution effect on the ISM physics, not on the measurement. $\to$ `noxray_analogues_kappa_degraded.{png,pdf,csv}`. Runs only where the reduced files exist (the cluster); skipped cleanly elsewhere. `NOX_P5B_LIMIT` caps the galaxy count for a smoke test.

In [ ]:
# ── Part 5b — kappa_gas of the m25 galaxies at m100 sampling: the estimator channel isolated ──
import h5py
P5B_DIR   = os.path.join(os.getcwd(), "output", "cis25", "reduced_particles")
P5B_FILE  = os.path.join(P5B_DIR, "snap_{snap:03d}", "m25n512_snap{snap:03d}_gal{gal:06d}.h5")
P5B_NREAL = 200                      # realisations of the degraded sample (each galaxy re-drawn)
P5B_M100  = 1.82e7                   # the m100 / m50 gas particle mass: n100 = member gas mass / this
P5B_LIMIT = int(os.environ.get("NOX_P5B_LIMIT", "0")) or None   # smoke test: only the first N galaxies
RNG5B = np.random.default_rng(11)


def kappa_rot(pos, vel, m):
    """Sales+10: K_rot / K about the particles' own total angular momentum, COM velocity frame."""
    if len(m) < 4 or m.sum() <= 0:
        return np.nan
    v = vel - np.average(vel, axis=0, weights=m)
    L = np.sum(m[:, None] * np.cross(pos, v), axis=0)
    if np.linalg.norm(L) == 0:
        return np.nan
    lhat = L / np.linalg.norm(L)
    jz = np.cross(pos, v) @ lhat
    R = np.linalg.norm(pos - np.outer(pos @ lhat, lhat), axis=1)
    ok = R > 0
    k = np.sum(m * np.sum(v ** 2, axis=1))
    return float(np.sum(m[ok] * (jz[ok] / R[ok]) ** 2) / k) if k > 0 else np.nan


if not os.path.isdir(P5B_DIR):
    print(f"{P5B_DIR} not found: the degradation experiment needs the m25 reduced particle files (cluster) — skipped")
else:
    g25b = QM[QM["box"] == "cis25"].reset_index(drop=True)
    if P5B_LIMIT:
        g25b = g25b.iloc[:P5B_LIMIT]
        print(f"NOX_P5B_LIMIT = {P5B_LIMIT}: smoke test on the first {len(g25b)} galaxies only")
    part, miss = {}, 0
    import time as _t; t0 = _t.time()
    for i, r in g25b.iterrows():
        p = P5B_FILE.format(snap=int(r["snap"]), gal=int(r["gal_id"]))
        if not os.path.exists(p):
            miss += 1; continue
        with h5py.File(p, "r") as f:
            mem = f["gas/member"][:].astype(bool)
            pos, vel, m = f["gas/pos"][:][mem], f["gas/vel"][:][mem], f["gas/m_gas"][:][mem]
        fin = np.isfinite(vel).all(axis=1) & np.isfinite(m) & (m > 0)
        part[i] = (pos[fin], vel[fin], m[fin])
        if (len(part) % 40 == 0):
            print(f"  loaded {len(part)} galaxies ({_t.time() - t0:.0f} s)", flush=True)
    print(f"member gas loaded for {len(part)} of {len(g25b)} galaxies ({miss} reduced files missing)")

    rows5b = []
    for i, (pos, vel, m) in part.items():
        r = g25b.loc[i]
        n100 = int(round(m.sum() / P5B_M100))
        kf = kappa_rot(pos, vel, m)
        ks = np.array([kappa_rot(pos[j], vel[j], m[j]) for j in
                       (RNG5B.choice(len(m), size=min(max(n100, 4), len(m)), replace=False) for _ in range(P5B_NREAL))])
        ks = ks[np.isfinite(ks)]
        rows5b.append(dict(snap=int(r["snap"]), gal_id=int(r["gal_id"]), n_mem=len(m), n100=n100, in_m100=n100 >= NGAS_MIN,
                           kappa_cat=float(r["kappa_gas"]), kappa_full=kf, kappa_sub=float(np.median(ks)) if len(ks) else np.nan,
                           kappa_sub_lo=float(np.percentile(ks, 16)) if len(ks) else np.nan, kappa_sub_hi=float(np.percentile(ks, 84)) if len(ks) else np.nan,
                           lfd=float(r["lfd"]), dusty=bool(r["lfd"] >= DUSTY_CUT)))
    D5B = pd.DataFrame(rows5b)
    KAP_CSV = os.path.join(OUT, "noxray_analogues_kappa_degraded.csv")
    D5B.to_csv(KAP_CSV, index=False)

    ok = np.isfinite(D5B["kappa_full"]) & np.isfinite(D5B["kappa_cat"])
    print(f"\nvalidation: kappa_full vs the catalogue kappa_gas over {int(ok.sum())} galaxies: "
          f"corr {np.corrcoef(D5B.loc[ok, 'kappa_full'], D5B.loc[ok, 'kappa_cat'])[0, 1]:.3f}, median |diff| {np.abs(D5B.loc[ok, 'kappa_full'] - D5B.loc[ok, 'kappa_cat']).median():.3f}")
    print(f"n100 (the particle count m100 would give the same reservoir): median {D5B['n100'].median():.0f} vs member {D5B['n_mem'].median():.0f}; "
          f"{int((~D5B['in_m100']).sum())} galaxies under the m100 {NGAS_MIN}-particle floor ({int((~D5B['in_m100'] & D5B['dusty']).sum())} of them dusty)")

    # the AUC per realisation: the whole sample, and the honest m100 selection (under-floor galaxies dropped)
    dusty_b = D5B["dusty"].to_numpy(bool)
    auc_cat, auc_full = auc_rank(D5B["kappa_cat"], dusty_b), auc_rank(D5B["kappa_full"], dusty_b)
    AUC_ALL, AUC_M100SEL = [], []
    for _ in range(P5B_NREAL):
        kd = np.full(len(D5B), np.nan)
        for row_j, (i, (pos, vel, m)) in enumerate(part.items()):
            n100 = D5B["n100"].to_numpy()[row_j]
            j = RNG5B.choice(len(m), size=min(max(int(n100), 4), len(m)), replace=False)
            kd[row_j] = kappa_rot(pos[j], vel[j], m[j])
        AUC_ALL.append(auc_rank(kd, dusty_b))
        keep = D5B["in_m100"].to_numpy(bool)
        AUC_M100SEL.append(auc_rank(kd[keep], dusty_b[keep]))
    AUC_ALL = np.array([v for v in AUC_ALL if np.isfinite(v)]); AUC_M100SEL = np.array([v for v in AUC_M100SEL if np.isfinite(v)])
    a100_obs = auc_rank(QM.loc[QM["box"] == "cis100", "kappa_gas"], QM.loc[QM["box"] == "cis100", "lfd"] >= DUSTY_CUT)
    print(f"\nAUC of kappa_gas for dust-rich on the SAME m25 galaxies:")
    print(f"  catalogue kappa          {auc_cat:.3f}")
    print(f"  recomputed, full N       {auc_full:.3f}")
    print(f"  degraded to m100 counts  {np.median(AUC_ALL):.3f} ({np.percentile(AUC_ALL, 16):.3f}-{np.percentile(AUC_ALL, 84):.3f})")
    print(f"  + the m100 selection     {np.median(AUC_M100SEL):.3f} ({np.percentile(AUC_M100SEL, 16):.3f}-{np.percentile(AUC_M100SEL, 84):.3f})  [under-floor galaxies dropped]")
    print(f"  m100's observed AUC      {a100_obs:.3f}")
    gap = auc_full - a100_obs
    closed = (auc_full - np.median(AUC_M100SEL)) / gap if gap > 0 else np.nan
    print(f"  -> the degradation closes {100 * closed:.0f} % of the m25 -> m100 gap: " +
          ("the ESTIMATOR at coarse sampling explains the box difference" if closed > 0.7 else
           "sampling noise alone does NOT explain the box difference — the remainder is the physics / population" if closed < 0.4 else
           "sampling noise explains part of the difference; the rest is the physics / population"))
    print(f"tables -> {KAP_CSV}")

    fig, axs = plt.subplots(1, 2, figsize=(10.4, 4.0))
    ax = axs[0]
    for d, mk, lab in [(D5B[~dusty_b], "o", "rest"), (D5B[dusty_b], "^", "dust-rich")]:
        ax.errorbar(d["kappa_full"], d["kappa_sub"], yerr=[d["kappa_sub"] - d["kappa_sub_lo"], d["kappa_sub_hi"] - d["kappa_sub"]],
                    fmt=mk, ms=5, color=BOX_STYLE["cis25"]["color"] if lab == "dust-rich" else "0.6", lw=0.7, alpha=0.75, label=lab)
    ax.plot([0, 1], [0, 1], color="0.3", ls=":", lw=1)
    ax.set_xlabel(r"$\kappa_{\rm gas}$ at full m25 sampling"); ax.set_ylabel(r"$\kappa_{\rm gas}$ at m100 sampling (median, 16-84 %)")
    ax.legend(fontsize=8); ax.grid(alpha=0.25, lw=0.5)
    ax = axs[1]
    ax.hist(AUC_ALL, bins=30, color="0.75", density=True, label="degraded, all galaxies")
    ax.hist(AUC_M100SEL, bins=30, histtype="step", color=BOX_STYLE["cis25"]["color"], lw=2, density=True, label="degraded + m100 selection")
    for v, c, lab in [(auc_full, "0.1", "full sampling"), (a100_obs, BOX_STYLE["cis100"]["color"], "m100 observed")]:
        ax.axvline(v, color=c, lw=2, ls="--"); ax.text(v, ax.get_ylim()[1] * 0.97, " " + lab, rotation=90, va="top", fontsize=7.6, color=c)
    ax.set_xlabel(r"AUC of $\kappa_{\rm gas}$ for dust-rich"); ax.set_ylabel("density"); ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=0.25, lw=0.5)
    fig.suptitle("the m25 sample re-measured at m100 sampling: is the estimator the bottleneck?", y=1.0)
    fig.tight_layout()
    paper_save(fig, "noxray_analogues_kappa_degraded")
    plt.show()

## Conventions, caveats, what the pieces are

* **Pure reads** — every input is a product of `paper_ism_prediction_boxes.ipynb` (Parts 1, 6a, 9). Re-running that notebook refreshes the inputs; this one never opens a catalogue.
* **Whole-galaxy (FOF) catalogue quantities** — as everywhere in the boxes notebook: no sightlines, no apertures; $R_{\rm e}$ = `R_PROJ_OVER_3D` $\times$ the 3-D stellar half-mass radius; zero dust sits at the `P9_FLOOR` on the log axes (it is the end point of the removal, never dropped).
* **s50nox has no histories** — the share holds only its anchor catalogues, so nox galaxies carry no $w_{\rm pre}$ / $E_x$ of their own (the ladder is ~20 GB at ROE `m50n512/s50nox/catalogs`; once downloaded, Part 6a of the boxes notebook builds them and `H_BOXES` already includes cis50nox). What we know of nox's own AGN state is the anchor: its jet-criterion fraction is printed in Part 1. The history question is answered on the *analogues*, which is where it is meaningful anyway — they live in runs where the X-ray channel exists.
* **cis50 histories cover only the $z \geq 1.15$ anchors** (the s50 ladder under snap 101 is complete, snaps 101–133 are not on the share): the Part 3 coverage per box is printed; cis100 is complete and carries the statistics.
* **Satellite chains inherit the central's record** — the main-branch chain (most-massive progenitor by stars) of a satellite can pass through its central, so a satellite's $w_{\rm pre}$ / $E_x$ can carry the bigger system's BH, $f_{\rm Edd}$ and $f_{\rm gas}$. Part 3 prints the fraction with an anchor BH below the jet threshold yet $E_x > 0$, treats satellite exposures as upper limits, and repeats the exposure contrasts on centrals and satellites separately; the anchor-state facts (BH not grown, the jet criterion, the never-exposed flag as an upper-limit statement) survive the caveat.
* **The match pools the anchors** — a nox galaxy at $z = 2$ and one at $z = 0.3$ enter one cloud; age and $\Delta \log R_{\rm e}$ absorb most of the epoch dependence ($\Delta \log R_{\rm e}$ is residual to a relation with a $\log(1+z)$ term), and the per-anchor analogue counts are printed so an anchor-driven match would show.
* **Part 5b needs the m25 reduced particle files** (`output/cis25/reduced_particles/`, the m25 particle sample: all 178 covered) — it runs on the cluster and skips cleanly where the files are absent; `NOX_P5B_LIMIT` caps the galaxy count for a smoke test. The full-sampling $\kappa$ is validated against the catalogue value in the printout before any conclusion is drawn.
* **Knobs** — `FEATURES` (default age + $\log M_{\rm dust}/M_\star$; the structural set is listed in the cell), `ANALOG_Q` (how far into the nox cloud's tail an analogue may sit), `MATCH_DM` (Part 1c pairing), `DISC_KAPPA`, `DUSTY_CUT` (Part 4), `ANALOG_BOXES` (cis50 / cis100 / cis25; the 25 Mpc gas floor is 8$\times$ lower, so its $\kappa_{\rm gas}$ / $f_{\rm gas}$ live on a different footing — its rows are a consistency check, not the statistics).